<a href="https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Click capture by position tier

The paper reports that weighted portfolio CTR declined from 0.423% in the top-three position tier to 0.050% in the deep tier. The paper states that these values were calculated as total clicks divided by total impressions within each position tier, rather than as a simple mean of page-level CTRs.

**My methodology question:** How were pages assigned to a position tier when their average position may change across the reporting window, and how much of the observed CTR difference remains after accounting for query intent, brand-query mix, content type, and SERP features? This question does not reject the finding. It asks how broadly the measured portfolio association can be transferred to another portfolio.

### Finding 2 — Freshness and refreshed mature content

The paper reports that mature content refreshed within 30 days had higher observed health and impression values than older stale content in the portfolio. It also notes that the 361+ day freshness bucket was too small and unstable to use as a headline result.

**My methodology question:** What exact event defines a page as refreshed, and how were refreshed mature pages compared with similar mature pages that were not refreshed? Pages selected for refresh may already have higher value, stronger demand, or more editorial attention. A time-aware comparison using pre-refresh performance and a comparable untreated group would help distinguish an observed association from an estimated refresh effect.

In [ ]:
import pandas as pd

paper_audit = pd.DataFrame([
    {
        "paper_finding": "Weighted CTR declines by position tier",
        "reported_measure": "0.423% top-three weighted CTR vs 0.050% deep-tier weighted CTR",
        "methodology_question": (
            "How are moving positions assigned to tiers, and are intent, "
            "query mix, content type, and SERP features accounted for?"
        )
    },
    {
        "paper_finding": "Refreshed mature content has higher observed performance",
        "reported_measure": "3.2x health and 57x impressions for the stated refreshed mature-content comparison",
        "methodology_question": (
            "What defines a refresh, and were refreshed pages compared with "
            "similar untreated pages using a time-aware design?"
        )
    }
])

display(paper_audit)

,paper_finding,reported_measure,methodology_question
0,Weighted CTR declines by position tier,0.423% top-three weighted CTR vs 0.050% deep-t...,"How are moving positions assigned to tiers, an..."
1,Refreshed mature content has higher observed p...,3.2x health and 57x impressions for the stated...,"What defines a refresh, and were refreshed pag..."


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Validation design

I will compare the same Logistic Regression model under two evaluation designs.

The first design is a random page-level split. It is useful as an optimistic reference, but pages from the same client can appear in both training and test data.

The second design is a grouped client holdout. Entire pseudonymized clients are assigned to either training or test data. This is more honest because it asks whether the model can rank pages for a client it did not see during training.

For both designs, model inputs come from 1–15 March 2026. The later period from 16–31 March 2026 is used only to construct the decline proxy. Precision@20, Precision@50, and the test-set base rate are reported for both splits.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [ ]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

print("Token loaded safely.")

Token loaded safely.


In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connected. Working month: March 2026")

DuckDB connected. Working month: March 2026


In [ ]:
feature_frame = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_impressions ELSE 0 END
            ) AS prior_15d_impressions,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_clicks ELSE 0 END
            ) AS prior_15d_clicks,

            AVG(
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN gsc_avg_position END
            ) AS prior_15d_avg_position,

            COUNT(DISTINCT
                CASE WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-15'
                THEN report_date END
            ) AS prior_15d_active_days,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-16'
                                     AND DATE '2026-03-31'
                THEN gsc_impressions ELSE 0 END
            ) AS next_16d_impressions

        FROM {MARCH}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        *,
        CASE
            WHEN prior_15d_impressions > 0
            THEN prior_15d_clicks / prior_15d_impressions
            ELSE 0
        END AS prior_15d_ctr,

        CASE
            WHEN prior_15d_impressions >= 100
             AND next_16d_impressions <= 0.80 * prior_15d_impressions
            THEN 1
            ELSE 0
        END AS is_next_15d_declining

    FROM page_windows
    WHERE prior_15d_impressions >= 100
""").df()

print(f"Feature-frame rows: {len(feature_frame):,}")
print(
    "Later-window decline proxy rate: "
    f"{feature_frame['is_next_15d_declining'].mean():.1%}"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 77,540
Later-window decline proxy rate: 28.6%


In [ ]:
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import sklearn

feature_columns = [
    "prior_15d_impressions",
    "prior_15d_clicks",
    "prior_15d_avg_position",
    "prior_15d_ctr",
    "prior_15d_active_days"
]

target_column = "is_next_15d_declining"

model_df = feature_frame.dropna(
    subset=feature_columns + [target_column, "client_hash_id"]
).copy()

X = model_df[feature_columns]
y = model_df[target_column]
groups = model_df["client_hash_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def make_logistic_model():
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ])

# Before: optimistic random page-level split.
random_train_idx, random_test_idx = train_test_split(
    np.arange(len(model_df)),
    test_size=0.25,
    random_state=42,
    stratify=y
)

random_model = make_logistic_model()

random_model.fit(
    X.iloc[random_train_idx],
    y.iloc[random_train_idx]
)

random_scores = random_model.predict_proba(
    X.iloc[random_test_idx]
)[:, 1]

# After: honest grouped client holdout.
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

group_train_idx, group_test_idx = next(
    group_split.split(X, y, groups=groups)
)

group_train_df = model_df.iloc[group_train_idx].copy()
group_test_df = model_df.iloc[group_test_idx].copy()

group_model = make_logistic_model()

group_model.fit(
    group_train_df[feature_columns],
    group_train_df[target_column]
)

group_scores = group_model.predict_proba(
    group_test_df[feature_columns]
)[:, 1]

shared_clients = set(
    group_train_df["client_hash_id"]
) & set(
    group_test_df["client_hash_id"]
)

assert len(shared_clients) == 0, (
    "A client appeared in both the grouped training and test data."
)

validation_comparison = pd.DataFrame([
    {
        "split_design": "Random page-level split",
        "Precision@20": precision_at_k(
            random_scores,
            y.iloc[random_test_idx],
            20
        ),
        "Precision@50": precision_at_k(
            random_scores,
            y.iloc[random_test_idx],
            50
        ),
        "test_base_rate": y.iloc[random_test_idx].mean(),
        "test_clients": "Clients overlap"
    },
    {
        "split_design": "Grouped client holdout",
        "Precision@20": precision_at_k(
            group_scores,
            group_test_df[target_column],
            20
        ),
        "Precision@50": precision_at_k(
            group_scores,
            group_test_df[target_column],
            50
        ),
        "test_base_rate": group_test_df[target_column].mean(),
        "test_clients": len(group_test_df["client_hash_id"].unique())
    }
])

print(f"scikit-learn version: {sklearn.__version__}")
print(
    "Grouped split: "
    f"{len(group_train_df):,} training rows, "
    f"{len(group_test_df):,} test rows, "
    f"{len(shared_clients)} shared clients."
)

display(validation_comparison.round(3))

scikit-learn version: 1.6.1
Grouped split: 61,747 training rows, 15,793 test rows, 0 shared clients.


,split_design,Precision@20,Precision@50,test_base_rate,test_clients
0,Random page-level split,0.75,0.66,0.286,Clients overlap
1,Grouped client holdout,0.15,0.28,0.148,10


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

The final model uses only five fields measured during the first 15 days of March 2026: prior impressions, clicks, average position, CTR, and active days.

The label is created from impressions observed between 16 and 31 March 2026. Therefore, `next_16d_impressions`, `is_next_15d_declining`, and any ratio derived from the later window are forbidden model inputs.

Pseudonymized client and content identifiers are context fields used for grouping, joining, and validation only. They are not model features. Existing product flags and previous decision scores are also excluded because they would reproduce an earlier decision rule rather than learn from observable signals.

In [ ]:
# Honest grouped-holdout score from the final feature set.
honest_group_scores = group_model.predict_proba(
    group_test_df[feature_columns]
)[:, 1]

# Deliberate future-window leakage test.
leaky_train = group_train_df[feature_columns].copy()
leaky_test = group_test_df[feature_columns].copy()

leaky_train["future_impression_ratio"] = (
    group_train_df["next_16d_impressions"]
    / group_train_df["prior_15d_impressions"]
)

leaky_test["future_impression_ratio"] = (
    group_test_df["next_16d_impressions"]
    / group_test_df["prior_15d_impressions"]
)

leaky_model = make_logistic_model()

leaky_model.fit(
    leaky_train,
    group_train_df[target_column]
)

leaky_scores = leaky_model.predict_proba(
    leaky_test
)[:, 1]

leakage_results = pd.DataFrame([
    {
        "feature_set": "Honest pre-decision features",
        "Precision@20": precision_at_k(
            honest_group_scores,
            group_test_df[target_column],
            20
        ),
        "Precision@50": precision_at_k(
            honest_group_scores,
            group_test_df[target_column],
            50
        )
    },
    {
        "feature_set": "Leaky set with future_impression_ratio",
        "Precision@20": precision_at_k(
            leaky_scores,
            group_test_df[target_column],
            20
        ),
        "Precision@50": precision_at_k(
            leaky_scores,
            group_test_df[target_column],
            50
        )
    }
])

feature_audit = pd.DataFrame([
    {
        "field": "prior_15d_impressions",
        "role": "feature",
        "available_at_decision_time": "Yes"
    },
    {
        "field": "prior_15d_clicks",
        "role": "feature",
        "available_at_decision_time": "Yes"
    },
    {
        "field": "prior_15d_avg_position",
        "role": "feature",
        "available_at_decision_time": "Yes"
    },
    {
        "field": "prior_15d_ctr",
        "role": "feature",
        "available_at_decision_time": "Yes"
    },
    {
        "field": "prior_15d_active_days",
        "role": "feature",
        "available_at_decision_time": "Yes"
    },
    {
        "field": "next_16d_impressions",
        "role": "excluded future outcome",
        "available_at_decision_time": "No"
    },
    {
        "field": "is_next_15d_declining",
        "role": "label only",
        "available_at_decision_time": "No"
    },
    {
        "field": "client_hash_id and content_hash_id",
        "role": "context and grouping only",
        "available_at_decision_time": "Not model inputs"
    }
])

print("Leakage comparison:")
display(leakage_results.round(3))

print("Final feature audit:")
display(feature_audit)

# Three real grouped-holdout error examples.
group_test_audit = group_test_df.copy()
group_test_audit["logistic_score"] = honest_group_scores
group_test_audit["predicted_class"] = (
    group_test_audit["logistic_score"] >= 0.50
).astype(int)

error_columns = [
    "content_hash_id",
    "prior_15d_impressions",
    "prior_15d_clicks",
    "prior_15d_avg_position",
    "prior_15d_ctr",
    "prior_15d_active_days",
    "logistic_score",
    "is_next_15d_declining"
]

false_positives = (
    group_test_audit[
        (group_test_audit["predicted_class"] == 1)
        & (group_test_audit[target_column] == 0)
    ]
    .sort_values("logistic_score", ascending=False)
    .head(3)
)

false_negatives = (
    group_test_audit[
        (group_test_audit["predicted_class"] == 0)
        & (group_test_audit[target_column] == 1)
    ]
    .sort_values("logistic_score", ascending=True)
    .head(3)
)

print("Three high-score false positives:")
display(false_positives[error_columns].round(4))

print("Three low-score false negatives:")
display(false_negatives[error_columns].round(4))

# Remove the deliberate leaky objects. They are not part of the final model.
del leaky_train, leaky_test, leaky_model, leaky_scores

Leakage comparison:


,feature_set,Precision@20,Precision@50
0,Honest pre-decision features,0.15,0.28
1,Leaky set with future_impression_ratio,1.00,1.00


Final feature audit:


,field,role,available_at_decision_time
0,prior_15d_impressions,feature,Yes
1,prior_15d_clicks,feature,Yes
2,prior_15d_avg_position,feature,Yes
3,prior_15d_ctr,feature,Yes
4,prior_15d_active_days,feature,Yes
5,next_16d_impressions,excluded future outcome,No
6,is_next_15d_declining,label only,No
7,client_hash_id and content_hash_id,context and grouping only,Not model inputs


Three high-score false positives:


,content_hash_id,prior_15d_impressions,prior_15d_clicks,prior_15d_avg_position,prior_15d_ctr,prior_15d_active_days,logistic_score,is_next_15d_declining
10017,content_ba462518dad435fc,44516.0,21.0,27.8556,0.0005,15,0.9034,0
2273,content_84a6bf3578312e90,41566.0,34.0,23.5860,0.0008,15,0.8399,0
2170,content_7a0a59b4cb181ab9,27860.0,15.0,33.7695,0.0005,15,0.8183,0


Three low-score false negatives:


,content_hash_id,prior_15d_impressions,prior_15d_clicks,prior_15d_avg_position,prior_15d_ctr,prior_15d_active_days,logistic_score,is_next_15d_declining
13997,content_747ff6b150f08f84,18917.0,217.0,2.7166,0.0115,13,0.0032,1
60613,content_7c47e5b8341f5d53,930.0,52.0,2.8631,0.0559,15,0.0150,1
27843,content_8ffa6201737fd638,106.0,8.0,3.8659,0.0755,15,0.0175,1


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Too strong:** “The Logistic Regression model predicts declining pages and should be used to decide which pages must be refreshed.”

**Evidence-aligned rewrite:** “In this March 2026 warehouse slice, Logistic Regression produced a higher observed Precision@50 than the hand-written baseline on one grouped client holdout. The model can support the ordering of a human review queue, but it does not prove that a page will decline, that an edit will restore performance, or that the result will generalize to every future client.”

In [ ]:
claim_rewrite = pd.DataFrame([
    {
        "claim_version": "Too strong",
        "claim": (
            "The Logistic Regression model predicts declining pages "
            "and should decide which pages must be refreshed."
        )
    },
    {
        "claim_version": "Evidence-aligned",
        "claim": (
            "The model showed higher observed Precision@50 than the "
            "baseline on one grouped client holdout and can support "
            "human review prioritisation."
        )
    }
])

display(claim_rewrite)

,claim_version,claim
0,Too strong,The Logistic Regression model predicts declini...
1,Evidence-aligned,The model showed higher observed Precision@50 ...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.